Goals: 
establish the following
* lap boundaries: the times when the car crosses the start/finish line
* lap count and lap times
* lap status: which laps are full timed laps, and which are out-laps (starting from standstill or the pit), in-laps (ending in the pit), or contain a common gap

* s goes 0 -> 805 and wraps back to 0 at the line
* so a lap boundary is where s suddenly drops by a large amount (from ~800 to ~0)
* lap time = the time between two consecutive boundaries

one of the issues we discovered in /notebooks/02_session.ipynb is:
- lap distance s flickers between ~805m and 0 when the car stops at the start/finish line (~975-980s), that produces lap reset so must be handled.


In [12]:
from src.loader import load
df = load("vehicle_curvilinear_coordinates")[["t_s", "s"]]

mask = (df["t_s"] >= 971) & (df["t_s"] <= 977)
mask2 = (df["s"] < 700)
flicker = df[mask & mask2]
segment = df[mask]

In [8]:
diff = flicker["s"].max() - flicker["s"].min()
diff

np.float64(6.2821958790210175)

In [13]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


fig = make_subplots(rows=1, cols=1, shared_xaxes=True)

fig.add_trace(
    go.Scattergl(
    x = segment["t_s"],
    y = segment["s"],
    name="s"
),
    row =1,
    col =1)

fig.update_yaxes(title_text="s", row=1, col=1)
fig.update_layout(
    height=300,
    width = 800,
    showlegend=True
)

fig.show()

hence, we need to think of what's physically true about a real line crossing that isn't true of the flicker

physically, we can follow two "rules":
1. a lap boundary is where s drops by more than half the track length AND the car is moving (wheel speed > motion_threshold_rad_s)
2. laps can't be impossibly short. min lap time = track length / the session's max speed

**assumption**: the car crosses the line driving forward. 


In [1]:
from src.utils import find_laps
find_laps()

,lap_number,start_s,end_s,lap_time_s,has_stop,has_gap,has_pit,is_out_lap,valid
0,1,61.447598,133.758730,72.311132,False,False,False,False,True
1,2,133.758730,281.465908,147.707178,True,False,False,False,False
2,3,281.465908,348.448507,66.982599,False,False,False,False,True
3,4,348.448507,413.994435,65.545928,False,False,False,False,True
4,5,413.994435,477.396409,63.401974,False,False,False,False,True
5,6,477.396409,539.871968,62.475559,False,False,False,False,True
6,7,539.871968,603.920050,64.048082,False,True,False,False,False
7,8,603.920050,665.375170,61.455120,False,False,False,False,True
8,9,665.375170,726.564443,61.189273,False,True,False,False,False
9,10,726.564443,787.367604,60.803161,False,True,False,False,False


Findings:
* there are 23 laps
* lap times: 61-67 s for normal laps, two long ones are explained by their flags (pit stop and stops)
* pit: lap 14, one lap spans it (972–1204s) with no boundaries inside and it's correctly flagged stop + gap + pit + out-lap
* gaps: laps 7, 9, 10, 11, 16 and 21 match the common gaps' times exactly (596, 689, 754, 809/814, 1306, 1621 s)
* valid laps: there are 8 in stint 1 (laps 1, 3–6, 8, 12, 13) and 7 in stint 2 (15, 17–20, 22, 23) which is enough to compare with fair balance.
* lap 1 is ~10s slower, to be investigated
* lap 15: first after leaving the pit, is also ~4-5s slower (67s) which is consistent with tyre behavior, to be investigated as well